# Lesson 4.4: GPT and Decoder Models

*Module 4 · Large Language Models*

> ChatGPT, Gemini, Claude — they all work the same way: predict the next word, add it, repeat. Let's build this process from scratch so you see exactly how AI writes.

---

## About this notebook

This notebook contains the **12 runnable code examples** from the published lesson `lesson-4.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — CLM — How GPT Learns (Causal Language Modeling) — `part1_clm_concept.py`
2. Step 2 — The Autoregressive Loop — Build It From Scratch — `part2_autoregressive.py — The generation loop`
3. Step 3 — Temperature — The Creativity Dial — `part3_temperature.py — Built from scratch`
4. Step 4 — Top-k and Top-p — Smart Filtering Before Picking — `part4_sampling.py — Top-k and Top-p from scratch`
5. Step 5 — Putting It All Together — Generate Text With Real GPT — `part5_generate_stories.py — See the magic!`
6. Step 6 — Same Concepts, Modern API — Gemini — `part6_gemini_generation.py`
7. Step 7 — The GPT Family — From 117M Parameters to Multimodal — `gpt_evolution.py`
8. Step 8 — Scaling Laws & Emergent Abilities — Why Bigger Models Are Smarter — `scaling_laws.py`
9. Step 9 — The 2025 Decoder Landscape — Open vs Closed Models — `decoder_landscape.py`
10. Step 10 — Logits & Logprobs — See What GPT Is Thinking — `logprobs_explorer.py`
11. Step 11 — From Raw Model to ChatGPT — The Training Pipeline — `training_pipeline.py`
12. Step 12 — Production Controls — Structured Outputs, Streaming, Chat Templates — `production_controls.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai transformers torch numpy


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export YOUR_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("YOUR_API_KEY", "YOUR_YOUR_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · CLM — How GPT Learns (Causal Language Modeling)

BERT guesses hidden words. GPT guesses the NEXT word. That's the key difference.

**`part1_clm_concept.py`** · _Block 1 of 12_

CLM: Causal Language Modeling — Given previous words → predict the next word.


In [ ]:
# =============================================
# CLM: Causal Language Modeling
# Given previous words → predict the next word.
# This is how GPT-4, Gemini, Claude all learn.
# =============================================

import numpy as np

# Imagine a tiny vocabulary
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran"]

# A very simple "language model" that just counts patterns
# After seeing "the", what usually comes next?
next_word_probs = {
    "the": {"cat": 0.4, "dog": 0.3, "mat": 0.3},
    "cat": {"sat": 0.6, "ran": 0.4},
    "dog": {"sat": 0.3, "ran": 0.7},
    "sat": {"on": 0.9, "the": 0.1},
    "on":  {"the": 0.95, "mat": 0.05},
}

# Generate text one word at a time!
current = "the"
sentence = [current]

print("Generating text word-by-word:\n")
for _ in range(5):
    if current not in next_word_probs:
        break
    
    options = next_word_probs[current]
    words = list(options.keys())
    probs = list(options.values())
    
    # Show the probabilities
    print(f"  After '{current}', options:")
    for w, p in zip(words, probs):
        bar = "#" * int(p * 20)
        print(f"    {w:6s} {p:.0%} {bar}")
    
    # Pick the next word randomly based on probabilities
    next_word = np.random.choice(words, p=probs)
    sentence.append(next_word)
    current = next_word
    print(f"  → Picked: '{next_word}'\n")

print(f"Generated: {' '.join(sentence)}")
print("\nRun it again — you'll get a different sentence!")
print("That's CLM: predict next word, pick one, repeat.")


> **What just happened?** We built a tiny language model using a dictionary of probabilities. After "the", it knows "cat" is 40% likely and "dog" is 30% likely. It picks one randomly (weighted by probability), then uses THAT word to pick the next. This is the exact same loop that GPT-4 uses — just with a neural network instead of a dictionary, and 100,000+ words instead of 7.


### Step 2 · The Autoregressive Loop — Build It From Scratch

The exact word-by-word generation process that powers every chatbot.

**`part2_autoregressive.py — The generation loop`** · _Block 2 of 12_

THE AUTOREGRESSIVE LOOP — This is EXACTLY how ChatGPT generates text.


In [ ]:
# =============================================
# THE AUTOREGRESSIVE LOOP
# This is EXACTLY how ChatGPT generates text.
# We'll do it step by step so you can see
# each word being chosen.
# =============================================

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GPT-2 (small version, runs on any laptop)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

# Our starting text (the "prompt")
prompt = "Once upon a time, there was a"

# Convert text to token IDs
input_ids = tokenizer.encode(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'\n")
print("Generating one word at a time:\n")

# Generate 20 new words, one at a time
for step in range(20):
    with torch.no_grad():
        # GPT looks at ALL previous tokens
        outputs = model(input_ids)
        
        # Get probabilities for the NEXT token only
        next_token_logits = outputs.logits[0, -1, :]  # last position
        probs = torch.softmax(next_token_logits, dim=-1)
        
        # Pick the most likely token (greedy)
        next_token_id = probs.argmax().unsqueeze(0).unsqueeze(0)
        next_word = tokenizer.decode(next_token_id[0])
        
        # Show what's happening
        confidence = probs.max().item()
        print(f"  Step {step+1:2d}: '{next_word}' ({confidence:.0%} confident)")
        
        # Add this token to the input for the next round
        input_ids = torch.cat([input_ids, next_token_id], dim=-1)

# Show the full generated text
full_text = tokenizer.decode(input_ids[0])
print(f"\nFull output:\n  '{full_text}'")


> **What just happened?** We manually ran the autoregressive loop that GPT uses: (1) Feed all previous words into the model. (2) Model outputs probabilities for the next word. (3) Pick the most likely word. (4) Add it to the input. (5) Repeat from step 1. Each step, the model sees one more word than before. After 20 rounds, we have 20 new words. That's text generation!


### Step 3 · Temperature — The Creativity Dial

Turn it down for facts. Turn it up for creativity.

**`part3_temperature.py — Built from scratch`** · _Block 3 of 12_

TEMPERATURE FROM SCRATCH — It's just ONE line of math. Let's see the


In [ ]:
# =============================================
# TEMPERATURE FROM SCRATCH
# It's just ONE line of math. Let's see the
# dramatic effect it has on word choice.
# =============================================

import numpy as np

def softmax(logits):
    """Convert scores to probabilities."""
    e = np.exp(logits - np.max(logits))
    return e / e.sum()

def apply_temperature(logits, temperature):
    """THE ENTIRE TRICK: divide by temperature, then softmax."""
    return softmax(logits / temperature)  # ← That's it. One division.

# GPT's raw scores for the next word after "The cat sat on the"
words  = ["mat", "floor", "roof", "moon", "pizza"]
logits = np.array([5.0,   4.2,    2.5,   0.8,   -1.0])

# Try different temperatures
print("Next word after 'The cat sat on the ___'\n")

for temp in [0.1, 0.5, 1.0, 2.0]:
    probs = apply_temperature(logits, temp)
    
    # What kind of temperature is this?
    if temp <= 0.3:   mood = "Very safe (always picks 'mat')"
    elif temp <= 0.7: mood = "Balanced (usually 'mat', sometimes 'floor')"
    elif temp <= 1.2: mood = "Creative (could pick 'roof' or 'moon')"
    else:             mood = "Wild! (might even pick 'pizza')"
    
    print(f"  Temperature = {temp}")
    print(f"  {mood}")
    for w, p in zip(words, probs):
        bar = "#" * int(p * 40)
        print(f"    {w:8s} {p:5.1%} {bar}")
    print()

# What you'll see:
# temp=0.1 → "mat" gets 99.9%, everything else ~0%
# temp=0.5 → "mat" ~65%, "floor" ~30%, others small
# temp=1.0 → "mat" ~40%, "floor" ~30%, "roof" ~15%
# temp=2.0 → Almost equal. Even "pizza" has ~8%!


> **What just happened?** 🧠 Knowledge Check What is the correct order for applying generation parameters? Top-p → temperature → top-k → sample Temperature → top-k → top-p → sample The order doesn't matter — they can be applied in any sequence Check Answer Temperature = dividing the scores by a number before softmax . That's the entire trick. • Small number (0.1) = scores get multiplied by 10, differences become HUGE, top word wins almost 100% • Large number (2.0) = scores get divided by 2, differences shrink, all words become roughly equal One line of math. Dramatic effect on creativity.


### Step 4 · Top-k and Top-p — Smart Filtering Before Picking

Don't just control randomness. Also filter out the garbage options first.

**`part4_sampling.py — Top-k and Top-p from scratch`** · _Block 4 of 12_

TOP-K and TOP-P SAMPLING — Filter out the bad options, THEN pick randomly


In [ ]:
# =============================================
# TOP-K and TOP-P SAMPLING
# Filter out the bad options, THEN pick randomly
# from the remaining good ones.
# =============================================

import numpy as np

words = ["mat", "floor", "roof", "moon", "pizza", "quantum", "banana"]
probs = np.array([0.35, 0.28, 0.18, 0.10, 0.05, 0.03, 0.01])

print("Original probabilities:")
for w, p in zip(words, probs):
    print(f"  {w:10s} {p:.0%} {'#' * int(p*30)}")

# ── TOP-K: Keep only the best K options ──
def top_k(words, probs, k=3):
    """Keep only the top k highest-probability words."""
    top_indices = np.argsort(probs)[-k:]  # indices of top k
    mask = np.zeros_like(probs)
    mask[top_indices] = probs[top_indices]
    mask /= mask.sum()  # re-normalize
    return mask

print("\nAfter Top-k = 3 (keep best 3, throw away rest):")
tk = top_k(words, probs, k=3)
for w, p in zip(words, tk):
    marker = " <-- kept" if p > 0 else " (removed)"
    print(f"  {w:10s} {p:.0%}{marker}")

# ── TOP-P: Keep words until cumulative prob reaches P ──
def top_p(words, probs, p=0.85):
    """Keep adding words until their total probability reaches p."""
    sorted_idx = np.argsort(probs)[::-1]  # best first
    cumulative = 0
    keep = []
    for idx in sorted_idx:
        keep.append(idx)
        cumulative += probs[idx]
        if cumulative >= p:
            break
    mask = np.zeros_like(probs)
    mask[keep] = probs[keep]
    mask /= mask.sum()
    return mask

print(f"\nAfter Top-p = 0.85 (keep until 85% covered):")
tp = top_p(words, probs, p=0.85)
cumul = 0
for w, p in zip(words, tp):
    if p > 0:
        cumul += p
        print(f"  {w:10s} {p:.0%}  (running total: {cumul:.0%})")
    else:
        print(f"  {w:10s}  -- (removed)")

# ── PICK A WORD using the filtered probabilities ──
print("\nSampling 10 words with Top-p = 0.85:")
for _ in range(10):
    choice = np.random.choice(words, p=tp)
    print(f"  → {choice}")


> **What just happened?** Top-k: We kept only the 3 best words and removed the rest. "pizza", "quantum", "banana" are gone — impossible to pick them now. Top-p: We kept adding words (best first) until their combined probability reached 85%. "mat" (35%) + "floor" (28%) + "roof" (18%) = 81%. Add "moon" (10%) = 91%. Stop. Only these 4 survive. Both methods prevent the model from picking absurd words while still allowing some creativity among the good options.


### Step 5 · Putting It All Together — Generate Text With Real GPT

Use Hugging Face to generate text with different settings and see the differences.

**`part5_generate_stories.py — See the magic!`** · _Block 5 of 12_

GENERATE TEXT with different settings — Same prompt, different parameters →


In [ ]:
# =============================================
# GENERATE TEXT with different settings
# Same prompt, different parameters → 
# completely different outputs!
# =============================================

from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

settings = [
    {"name": "Greedy (boring but safe)",
     "args": {"do_sample": False}},
    
    {"name": "Low temperature (factual)",
     "args": {"do_sample": True, "temperature": 0.3, "top_k": 50}},
    
    {"name": "Medium temperature (balanced)",
     "args": {"do_sample": True, "temperature": 0.7, "top_p": 0.9}},
    
    {"name": "High temperature (creative)",
     "args": {"do_sample": True, "temperature": 1.2, "top_p": 0.95}},
    
    {"name": "Very high temperature (wild!)",
     "args": {"do_sample": True, "temperature": 1.8, "top_k": 100}},
]

print(f"Prompt: '{prompt}'\n")
print("=" * 60)

for s in settings:
    output = model.generate(
        input_ids,
        max_new_tokens=40,
        pad_token_id=tokenizer.eos_token_id,
        **s["args"]
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    print(f"\n{s['name']}:")
    print(f"  {text}\n")
    print("-" * 60)

print("""
Notice how:
  - Greedy: repetitive, safe, always the same output
  - Low temp: coherent and focused, slight variation
  - Medium: good balance of creativity and coherence
  - High: more creative but sometimes odd word choices
  - Very high: often produces nonsense or bizarre tangents

This is why prompt engineering matters — even the SAME model
gives wildly different results with different settings!
""")


> **What just happened?** We gave GPT-2 the exact same prompt 5 times with different generation settings. Same model, same prompt, completely different outputs. Temperature controls how "spicy" the word choices are. Top-k and top-p control how many options the model considers. This is why understanding these parameters matters — they're the knobs you'll tune in every GenAI application you build.


### Step 6 · Same Concepts, Modern API — Gemini

Now that you understand what's under the hood, use a production API.

**`part6_gemini_generation.py`** · _Block 6 of 12_

SAME CONCEPTS WITH GEMINI API — Now you know what these parameters DO under


In [ ]:
# =============================================
# SAME CONCEPTS WITH GEMINI API
# Now you know what these parameters DO under
# the hood — use them with confidence!
# =============================================

import google.generativeai as genai

genai.configure(api_key="YOUR_API_KEY")

prompt = "Write a one-paragraph story about a robot learning to cook."

configs = [
    ("Safe & factual",     0.2, 0.5),   # (name, temperature, top_p)
    ("Balanced",           0.7, 0.9),
    ("Creative",           1.2, 0.95),
]

for name, temp, top_p in configs:
    model = genai.GenerativeModel(
        "gemini-2.0-flash",
        generation_config={
            "temperature": temp,     # You now know: logits / temp
            "top_p": top_p,           # You now know: keep until cumulative prob
            "max_output_tokens": 150,
        }
    )
    
    response = model.generate_content(prompt)
    
    print(f"\n{'='*50}")
    print(f"{name} (temp={temp}, top_p={top_p})")
    print(f"{'='*50}")
    print(response.text)

print("""
You now understand EXACTLY what these parameters do:

  temperature=0.2 → divides scores by 0.2 → top word dominates
  temperature=1.2 → divides scores by 1.2 → more words get a chance
  top_p=0.5 → only considers words covering 50% of probability
  top_p=0.95 → considers words covering 95% of probability

You're no longer blindly copy-pasting parameters.
You KNOW what they do. That's the power of understanding.
""")


> **What just happened?** 🧠 Knowledge Check Why did a 1.3B InstructGPT outperform the 175B GPT-3? InstructGPT used a better transformer architecture RLHF alignment (training on human preferences) matters more than raw model size for helpfulness The 175B GPT-3 was poorly trained with bad data Check Answer The same temperature and top-p concepts you built from scratch in Steps 3-4 work identically in production APIs. The only difference: GPT-2 runs locally (free, 1.5B params), while Gemini runs in the cloud (paid, much larger). Every GenAI API you'll ever use — OpenAI, Anthropic, Gemini, Mistral — has these exact same knobs. You now understand all of them.


### Step 7 · The GPT Family — From 117M Parameters to Multimodal

Each generation added a key insight. Understanding the evolution explains why ChatGPT works.

**`gpt_evolution.py`** · _Block 7 of 12_

THE GPT FAMILY — What changed at each step


In [ ]:
# =============================================
# THE GPT FAMILY — What changed at each step
# =============================================

gpt_family = [
    ("GPT-1",    2018, "117M",   "Proved: decoder pre-training + fine-tuning works"),
    ("GPT-2",    2019, "1.5B",   "Proved: scale → zero-shot abilities (no fine-tuning!)"),
    ("GPT-3",    2020, "175B",   "Proved: in-context learning (few-shot from prompt)"),
    ("InstructGPT",2022,"~1.3B", "Added RLHF → follows instructions, less harmful"),
    ("ChatGPT",  2022, "~20B?",  "InstructGPT + dialogue fine-tuning → the product"),
    ("GPT-4",    2023, "~1.8T?", "MoE architecture, multimodal (text + images)"),
    ("GPT-4o",   2024, "smaller","Omni: text + images + audio, 2x cheaper, 2x faster"),
    ("o1/o3",    2024, "unknown","Chain-of-thought reasoning, test-time compute scaling"),
]

print(f"{'Model':<14} {'Year':>5} {'Params':>8} Key Innovation")
print("-" * 75)
for name, year, params, innovation in gpt_family:
    print(f"  {name:<12} {year:>5} {params:>8} {innovation}")

print("""
THE KEY TRANSITIONS:
  GPT-1 → GPT-2:  Scale unlocks zero-shot abilities (no fine-tuning needed)
  GPT-2 → GPT-3:  In-context learning (put examples IN the prompt)
  GPT-3 → ChatGPT: RLHF alignment (Module 6) makes it follow instructions
  ChatGPT → GPT-4: MoE + multimodal (images, audio) → Module 7
  GPT-4 → o1/o3:  Test-time compute (think longer → better answers)

The training pipeline (preview of Module 6):
  1. Pre-training:   Predict next token on internet text ($$$$)
  2. SFT:            Fine-tune on instruction-response pairs
  3. RLHF/DPO:       Human preference alignment → helpful, harmless, honest
  4. Reasoning:       Chain-of-thought fine-tuning (o1/o3)
""")


> **What just happened?** Each GPT generation added one key insight: GPT-1 proved pre-training works. GPT-2 proved scale unlocks zero-shot. GPT-3 proved in-context learning. InstructGPT/ChatGPT proved RLHF alignment. GPT-4 proved MoE + multimodal. o1/o3 proved test-time compute. The training pipeline — pre-train → SFT → RLHF → reasoning — is covered in Module 6. Understanding this history explains WHY modern APIs work the way they do.


### Step 8 · Scaling Laws & Emergent Abilities — Why Bigger Models Are Smarter

Performance follows predictable power laws. But some abilities appear suddenly at scale.

**`scaling_laws.py`** · _Block 8 of 12_

SCALING LAWS — How size → performance


In [ ]:
# =============================================
# SCALING LAWS — How size → performance
# =============================================

# Kaplan Scaling Laws (2020) — OpenAI
# "Performance follows power law with model size, data, compute"
# 10x more compute → scale model 5.5x, data 1.8x
# This led to GPT-3: 175B params but only 300B tokens (undertrained!)

# Chinchilla Scaling Laws (2022) — DeepMind
# "Data matters more than we thought!"
# Optimal: 20 tokens per parameter
# GPT-3 should have been either 20x smaller OR trained on 20x more data

scaling = [
    ("GPT-3",     175,   300,    1.7,   "Undertrained by Chinchilla standards"),
    ("Chinchilla", 70,   1400,   20.0,  "Optimal: matched GPT-3 quality at 2.5x less cost"),
    ("LLaMA 1",   7,     1000,   142.0, "Overtrained: better at inference (cheaper to serve)"),
    ("LLaMA 3",   8,     15000,  1875.0,"Massively overtrained: best 8B model ever"),
]

print(f"{'Model':<12} {'Params(B)':>10} {'Tokens(B)':>10} {'Tok/Param':>10} Strategy")
print("-" * 75)
for name, params, tokens, ratio, desc in scaling:
    print(f"  {name:<10} {params:>10} {tokens:>10} {ratio:>10.1f} {desc}")

print("""
EMERGENT ABILITIES — capabilities that "switch on" at scale:
  ~13B params: Few-shot arithmetic, word unscrambling
  ~62B params: Chain-of-thought reasoning
  ~175B params: In-context learning, code generation
  
  But recent research (2023-2025) shows: "emergence" may be a
  measurement artifact — smooth improvement looks discontinuous
  when using binary (exact match) metrics. With continuous metrics,
  improvements are gradual. Still, scale clearly helps.

IN-CONTEXT LEARNING — GPT's superpower:
  Prompt: "Translate: cat→gato, dog→perro, house→"
  Output: "casa"
  
  GPT learned translation from 2 examples IN THE PROMPT.
  No weight updates. No fine-tuning. Just pattern recognition
  from pre-training on billions of text patterns.
  This is why prompt engineering (Module 5) is so powerful.
""")


> **What just happened?** Two scaling law eras: Kaplan (2020) said "make models bigger." Chinchilla (2022) said "use more data." LLaMA 3 took this further: 8B params trained on 15 TRILLION tokens — 1,875x more data than the "optimal" ratio. This "overtraining" makes small models cheaper to serve. Emergent abilities like in-context learning (learning from prompt examples without weight updates) appear at scale — this is why prompt engineering works and why it's covered in Module 5.


### Step 9 · The 2025 Decoder Landscape — Open vs Closed Models

The models you'll use in production. Know what's available, what it costs, and when to use each.

**`decoder_landscape.py`** · _Block 9 of 12_

2025 DECODER MODEL LANDSCAPE


In [ ]:
# =============================================
# 2025 DECODER MODEL LANDSCAPE
# =============================================

models = [
    # (Name, Provider, Type, Size, $/M input, $/M output, Notes)
    ("GPT-4o",        "OpenAI",    "Closed", "~200B?",  2.50,  10.00, "Best ecosystem, function calling"),
    ("Claude Sonnet", "Anthropic", "Closed", "unknown", 3.00,  15.00, "Best for code, long context"),
    ("Gemini Flash",  "Google",    "Closed", "unknown", 0.15,  0.60,  "Cheapest, 1M context, multimodal"),
    ("LLaMA 3 70B",  "Meta",      "Open",   "70B",     0.00,  0.00,  "Free, self-host, fine-tunable"),
    ("Mistral Large","Mistral",   "Open",   "123B",    2.00,  6.00,  "European, strong reasoning"),
    ("DeepSeek V3",  "DeepSeek",  "Open",   "671B MoE",0.27, 1.10,  "37B active, cheapest smart model"),
    ("Qwen 2.5 72B","Alibaba",   "Open",   "72B",     0.00,  0.00,  "Multilingual, Apache 2.0"),
    ("Gemma 3 27B", "Google",    "Open",   "27B",     0.00,  0.00,  "Best small model, runs on laptop"),
]

print(f"{'Model':<16} {'Provider':<11} {'Type':<7} {'Size':<10} {'$/M In':>7} {'$/M Out':>8}")
print("-" * 70)
for name, prov, typ, size, inp, out, _ in models:
    print(f"  {name:<14} {prov:<11} {typ:<7} {size:<10} ${inp:>6.2f} ${out:>7.2f}")

print("""
DECISION FRAMEWORK (Module 13 goes deeper):
  Budget app:     Gemini Flash ($0.15/M) or DeepSeek V3 ($0.27/M)
  Best quality:   GPT-4o or Claude for complex reasoning
  Self-hosted:    LLaMA 3 70B or Qwen 2.5 72B (free, your GPUs)
  Edge/mobile:    Gemma 3 27B or LLaMA 3 8B (quantized to 4-bit)
  Indian langs:   Gemini (best Hindi), Qwen (good multilingual)
  Fine-tuning:    LLaMA 3 or Mistral (Module 9)

  Open models are FREE to download, run, and fine-tune.
  Closed models charge per-token but require no infrastructure.
  DeepSeek V3: 671B total params, only 37B active (MoE from Lesson 4.4!)
""")


> **What just happened?** 🧠 Knowledge Check What is structured output and how does constrained decoding work? The model is fine-tuned to always output JSON At each generation step, tokens that would violate the JSON schema get probability 0, so the model CANNOT produce invalid output The output is parsed and corrected after generation Check Answer The 2025 decoder landscape: Closed models (GPT-4o, Claude, Gemini) offer convenience at per-token cost. Open models (LLaMA, Mistral, DeepSeek, Qwen, Gemma) are free to download, self-host, and fine-tune. DeepSeek V3 uses MoE (from Lesson 4.4) — 671B total params but only 37B active per token, making it incredibly cost-effective. Module 9 (fine-tuning) and Module 13 (deployment) build directly on this landscape.


### Step 10 · Logits & Logprobs — See What GPT Is Thinking

The most powerful debugging tool: see the probability of every token at every step.

**`logprobs_explorer.py`** · _Block 10 of 12_

LOGITS → LOGPROBS — See inside the model's head


In [ ]:
# =============================================
# LOGITS → LOGPROBS — See inside the model's head
# =============================================
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

prompt = "The capital of India is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]  # Last position's raw scores

# Logits → probabilities → logprobs
probs = F.softmax(logits, dim=-1)
logprobs = torch.log(probs)

# Top 5 most likely next tokens
top5 = torch.topk(probs, 5)
print("What GPT thinks comes after 'The capital of India is':\n")
for i in range(5):
    token = tokenizer.decode(top5.indices[i])
    prob = top5.values[i].item()
    lp = logprobs[top5.indices[i]].item()
    print(f"  {i+1}. '{token}' → prob={prob:.1%}, logprob={lp:.2f}")

print("""
LOGITS vs LOGPROBS vs PROBABILITIES:
  logits  = raw scores from model (unbounded, e.g. -2.1, 5.3, 1.7)
  probs   = softmax(logits) → sums to 1.0 (e.g. 0.45, 0.30, 0.15)
  logprobs = log(probs) → always negative (e.g. -0.80, -1.20, -1.90)
  
  logprob = 0.0  → 100% confident (never happens in practice)
  logprob = -0.7 → ~50% confident
  logprob = -2.3 → ~10% confident
  logprob = -4.6 → ~1% confident

PRODUCTION USES of logprobs:
  1. Hallucination detection → low logprob on factual claims = flag for review
  2. Classification confidence → use logprobs instead of text parsing
  3. Perplexity = exp(-avg logprob) → standard metric for model quality
     Lower perplexity = model is more certain = better model
  4. Token cost optimization → route low-confidence queries to larger models
""")


> **What just happened?** You looked inside GPT's decision process. At each generation step, the model produces logits (raw scores for all 50,257 tokens), which become probabilities via softmax, and logprobs via log(). This is the same pipeline your temperature and top-p code modifies! Perplexity = exp(-avg logprob) is THE standard metric for evaluating language models. In production, logprobs enable hallucination detection, confidence-based routing, and classification without text parsing.


### Step 11 · From Raw Model to ChatGPT — The Training Pipeline

The most important story in AI: how a 1.3B InstructGPT beat a 175B GPT-3.

**`training_pipeline.py`** · _Block 11 of 12_

THE LLM TRAINING PIPELINE — 4 stages


In [ ]:
# =============================================
# THE LLM TRAINING PIPELINE — 4 stages
# =============================================

pipeline = {
    "1. Pre-Training": {
        "task": "Next-token prediction on trillions of tokens",
        "data": "Internet text (FineWeb, Common Crawl, books, code)",
        "cost": "$10K-$100M+ (most expensive stage)",
        "output": "Base model — completes text but doesn't follow instructions",
        "example": "'What is 2+2?' → 'What is 2+3? What is 2+4?...' (just continues)",
    },
    "2. SFT (Supervised Fine-Tuning)": {
        "task": "Train on (instruction, ideal_response) pairs",
        "data": "~100K human-written instruction-response examples",
        "cost": "$100-$10K",
        "output": "Instruction-following model — answers questions properly",
        "example": "'What is 2+2?' → '4' (follows instruction format)",
    },
    "3. RLHF / DPO (Alignment)": {
        "task": "Optimize for human preferences",
        "data": "Human rankings: 'Response A is better than B'",
        "cost": "$10K-$1M (human annotators are expensive)",
        "output": "Aligned model — helpful, harmless, honest",
        "methods": "PPO → DPO → GRPO (each simpler than the last)",
    },
    "4. Reasoning (2024+)": {
        "task": "Chain-of-thought fine-tuning + test-time compute",
        "data": "Reasoning traces (verified math, code, logic)",
        "cost": "40%+ of total compute budget",
        "output": "Reasoning model — thinks before answering (o1, o3, R1)",
        "key": "DeepSeek R1: GRPO only, no SFT needed for reasoning!",
    },
}

for stage, info in pipeline.items():
    print(f"\n{stage}")
    for k, v in info.items():
        print(f"  {k}: {v}")

print("""
THE INSIGHT: Alignment beats scale.
  InstructGPT (1.3B params + RLHF) was PREFERRED over GPT-3 (175B).
  A small, well-aligned model > a huge, unaligned model.

ALIGNMENT EVOLUTION (each removes a requirement):
  RLHF (2022): Reward model + PPO → 4 models needed
  DPO  (2023): Direct preference → 2 models (no reward model)
  GRPO (2025): Group relative policy → no critic model, 33% less memory
  ORPO (2024): Single unified objective → 1 model
  
Module 6 covers SFT + RLHF/DPO in depth with hands-on labs.
""")


> **What just happened?** The LLM training pipeline has four stages: pre-training (learn language), SFT (learn to follow instructions), RLHF/DPO (learn human preferences), and reasoning (learn to think step-by-step). The key insight: a 1.3B InstructGPT with RLHF was preferred over the 175B GPT-3 — alignment beats scale. Each new method (PPO→DPO→GRPO→ORPO) is simpler than the last. Module 6 goes deep on stages 2-3.


### Step 12 · Production Controls — Structured Outputs, Streaming, Chat Templates

The three features you'll use daily as a GenAI engineer.

**`production_controls.py`** · _Block 12 of 12_

PRODUCTION CONTROLS — What engineers use daily


In [ ]:
# =============================================
# PRODUCTION CONTROLS — What engineers use daily
# =============================================
import google.generativeai as genai

# 1. CHAT TEMPLATES — system/user/assistant format
# Under the hood, this is just concatenated text for next-token prediction:
#   <system>You are a helpful assistant.</system>
#   <user>What is 2+2?</user>
#   <assistant>4</assistant>
# The model does CLM on this ENTIRE sequence — same as Step 1!

model = genai.GenerativeModel(
    "gemini-2.0-flash",
    system_instruction="You are a concise technical assistant. Reply in JSON.",
)

# 2. STRUCTURED OUTPUTS — force valid JSON
# Internally: constrained decoding masks non-conforming tokens at each step
# If the schema says "age" must be an integer, tokens like "twenty" get
# probability 0.0 — the model CANNOT produce invalid output.

response = model.generate_content(
    "Extract: 'Sanjay Kumar, age 35, lives in Hyderabad'",
    generation_config={
        "response_mime_type": "application/json",
        "response_schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "integer"},
                "city": {"type": "string"},
            }
        }
    }
)
print(f"Structured output: {response.text}")
# → {"name": "Sanjay Kumar", "age": 35, "city": "Hyderabad"}

# 3. STREAMING — token-by-token delivery (SSE)
# Why streaming? 34% longer session times in user studies.
# Each token is sent as it's generated via Server-Sent Events.
for chunk in model.generate_content("Explain RAG in 3 bullets", stream=True):
    print(chunk.text, end="", flush=True)  # tokens appear one by one

print("""

OTHER PRODUCTION PARAMETERS:
  repetition_penalty: 1.2  → penalizes tokens already generated (prevents loops)
  frequency_penalty:  0.5  → reduces probability of frequent tokens (OpenAI-specific)
  presence_penalty:   0.5  → encourages topic diversity (OpenAI-specific)
  stop_sequences: ["\\n"]  → stop generating when this token appears
  max_tokens: 500          → hard limit on output length
  
DECODING STRATEGIES (beyond temp/top-k/top-p):
  Greedy:       do_sample=False → always pick highest prob (deterministic)
  Beam search:  num_beams=4 → explore multiple paths (best for translation)
  Contrastive:  penalty_alpha → balances quality and diversity
  Min-p:        min_p=0.1 → dynamic threshold (popular in local LLM community)
""")


> **What just happened?** Three production essentials: Chat templates (system/user/assistant) are just concatenated text for the same CLM you learned in Step 1 — no magic. Structured outputs use constrained decoding (masking invalid tokens at each step) to guarantee valid JSON — essential for APIs and agents. Streaming via Server-Sent Events delivers tokens one-by-one, improving UX by 34%. Plus: repetition penalty, beam search, and stop sequences complete your production toolkit. Module 10 (agents) relies heavily on structured outputs for tool calling.


---

## Wrap-up

You've walked through all 12 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
